# Model Context Protocol (MCP) Server Example
This Jupyter notebook runs on Colab and demonstrates a MCP server running Mellea.

### How Model Context Protocol (MCP) Works and Its Importance

The Model Context Protocol (MCP) is designed to standardize the way **tools** and **resources** are exposed and consumed by **agents**, particularly large language models (LLMs). It provides a structured interface that allows agents to discover and interact with external functionalities.

#### 1. How MCP Works:

*   **Standardized Interface:** MCP defines a common language for tools and agents to communicate. Instead of an agent needing to know the specific API or function signature for every tool, it interacts with a generalized MCP interface.
*   **Tool/Resource Definition:** Developers define tools (functions that perform actions, often involving LLMs) and resources (data providers or simple utilities) and register them with an MCP server. This involves providing clear descriptions, input parameters, and expected output types.
*   **Schema Exposure:** The MCP server exposes the schemas (descriptions) of all registered tools and resources. This allows an agent to understand what capabilities are available, what inputs they require, and what outputs they produce, without needing to inspect the underlying code.
*   **Abstraction:** MCP abstracts away the implementation details of the tools. An agent doesn't need to know if a tool makes an API call, runs a local script, or interacts with a database; it only needs to know how to call the MCP-defined interface.

#### 2. How it Runs:

*   **MCP Server:** An MCP server (like `FastMCP` used in the example) acts as a central registry and dispatcher. It runs, listens for requests, and manages the lifecycle of registered tools and resources.
*   **Registration:** Developers use decorators (e.g., `@mcp.tool()`, `@mcp.resource()`) to register Python functions or classes as MCP tools or resources with the server. This registration process makes them discoverable.
*   **Discovery:** An MCP client (which could be an LLM agent, another program, or even a human) can query the server to discover the list of available tools and resources, along with their metadata (names, descriptions, parameters).
*   **Invocation:** Once an agent identifies a suitable tool or resource, it sends a standardized invocation request to the MCP server, providing the necessary arguments. The server then executes the corresponding backend function and returns the result to the agent.
*   **Example from Notebook:** In the provided notebook, `FastMCP("ColabMistralDemo")` initializes the server. `write_a_poem_with_api` is registered as a tool, and `get_greeting` as a resource. The subsequent `try-except` blocks simulate how an agent might discover and invoke these functionalities.

#### 3. What Without MCP We Can't Do (or would be harder):

Without a protocol like MCP, several key functionalities and benefits for agents and tool integration would be significantly more challenging or impossible:

*   **Dynamic Tool Discovery and Use:** Agents would struggle to dynamically find and utilize new tools without being explicitly programmed or retrained for each new capability. MCP enables agents to browse and adapt to a constantly evolving set of tools.
*   **Enhanced Interoperability:** Different systems, agents, or even different LLMs would find it difficult to seamlessly communicate and share functionalities. MCP provides a universal adapter.
*   **Improved Agent Autonomy:** An agent with MCP can independently expand its capabilities by integrating new tools, leading to more autonomous and versatile AI systems.
*   **Simplified Tool Integration for Developers:** Developers can expose their functions as tools without needing to worry about how agents will specifically consume them, as MCP handles the interface standardization.
*   **Scalability and Maintainability:** Managing a growing number of tools and ensuring their compatibility with various agents becomes much more scalable and maintainable when mediated by a standard protocol. Without it, every new tool or agent could require bespoke integration logic.

In [6]:
# -*- coding: utf-8 -*-
"""
MCP Example for Google Colab - Using MistralAI API (Simulated Structure)
=======================================================================

This script demonstrates defining an MCP server using Mellea.
It simulates how a tool might integrate with an external LLM API like MistralAI,
instead of relying on a local Ollama service, which is problematic in Colab.
The MistralAI integration is simulated for structural clarity.
"""

# --- Step 1: Install Required Libraries (Mellea + LangChain for API simulation) ---
print("--- Installing Libraries ---")
import os
# Install Mellea
os.system('pip install -q mcp mellea')
# Install LangChain components for MistralAI (assuming the code snippet provided is the correct way)
os.system('pip install -q langchain-mistralai langchain-core langchain-community')

# --- Step 2: Import Libraries ---
print("--- Importing Libraries ---")
from mcp.server.fastmcp import FastMCP
# Import MistralAI components (assuming they were installed correctly)
try:
    from langchain_mistralai import ChatMistralAI
    from google.colab import userdata # For accessing Colab secrets
    print("LangChain MistralAI components imported.")
    LANGCHAIN_AVAILABLE = True
except ImportError:
    print("LangChain MistralAI not available. Proceeding with simulated API call structure.")
    LANGCHAIN_AVAILABLE = False


# --- Step 3: Setup API Key (Conditional) ---
# Retrieve the API key from Colab secrets if LangChain is available.
MISTRAL_API_KEY = None
if LANGCHAIN_AVAILABLE:
    try:
        MISTRAL_API_KEY = userdata.get("Mistral_API") # Replace "Mistral_API" with your actual secret name
        if not MISTRAL_API_KEY:
            print("WARNING: 'Mistral_API' secret not found in Colab. API calls will fail.")
    except Exception as e:
        print(f"WARNING: Could not retrieve 'Mistral_API' secret: {e}. API calls will fail.")


# --- Step 4: Define MCP Server and Components ---
print("--- Defining MCP Server and Components ---")
mcp = FastMCP("ColabMistralDemo")

# Define an MCP Tool that SIMULATES using an external LLM API (e.g., MistralAI).
# This function structure shows how you'd integrate the API call.
@mcp.tool()
def write_a_poem_with_api(word_limit: int) -> str:
    """
    An MCP tool that generates a short poem using an external LLM API (simulated here).
    This demonstrates the structure for calling MistralAI, bypassing Mellea's Ollama backend.
    """
    if not LANGCHAIN_AVAILABLE:
        return "Error: LangChain MistralAI not installed."

    if not MISTRAL_API_KEY:
         return "Error: MistralAI API key not found in Colab secrets."

    # --- Simulate or Execute the API Call ---
    # In a real scenario, you would initialize the LLM and call it here.
    # Example initialization (requires valid key):
    # llm = ChatMistralAI(model='mistral-small-latest', api_key=MISTRAL_API_KEY, temperature=0.7)
    # prompt = f"Write a short, creative poem about technology, strictly less than {word_limit} words."
    # response = llm.invoke(prompt)
    # poem = response.content # Or however the output is accessed

    # For this demo, since we might not have a key, we simulate the call.
    # Replace the simulation block below with the actual API call code.
    print(f"  [SIMULATION] Calling MistralAI API to generate a poem with <= {word_limit} words.")
    # Simulate a successful API response
    simulated_poem = f"This is a simulated poem generated via MistralAI API, constrained to {word_limit} words or fewer. Technology flows like streams of light."
    print(f"  [SIMULATION] Received response from API.")
    return simulated_poem
    # End of simulation block


# Define an MCP Resource that does NOT require an LLM.
# This part CAN work within Colab as it's just a function.
@mcp.resource("greeting://{name}")
def get_greeting(name: str) -> str:
    """
    An MCP resource that returns a personalized greeting.
    This works independently of any LLM backend.
    """
    return f"Hello, {name}! This greeting resource works in Colab using MistralAI as a potential LLM source."

# --- Step 5: Test the Defined Components (Simulated Execution) ---
print("--- Testing MCP Components (Simulated Execution) ---")

# Test the 'get_greeting' resource (This SHOULD work).
print("\n--- Calling 'get_greeting' Resource ---")
try:
    greeting_message = get_greeting(name="Colab User")
    print("Generated Greeting:")
    print(greeting_message)
except Exception as e:
    print(f"An error occurred while calling 'get_greeting': {e}")

# Test the 'write_a_poem_with_api' tool (This will simulate the API call).
print("\n--- Calling 'write_a_poem_with_api' Tool (Simulating API) ---")
try:
    generated_poem = write_a_poem_with_api(word_limit=15)
    print("Generated Poem (<= 15 words, simulated API call):")
    print(generated_poem)
except Exception as e:
    print(f"An error occurred while calling 'write_a_poem_with_api': {e}")


print("\n--- MCP Server Definition Complete ---")
print("This script defines an MCP server capable of using an external LLM API like MistralAI.")
print("The actual API call within 'write_a_poem_with_api' needs a valid API key and proper LangChain setup.")
print("Remember to add your MistralAI API key to Colab secrets as 'Mistral_API'.")
print("The server definition is complete; an external MCP client would now be able to discover and invoke these tools/resources.")


--- Installing Libraries ---
--- Importing Libraries ---
LangChain MistralAI components imported.
--- Defining MCP Server and Components ---
--- Testing MCP Components (Simulated Execution) ---

--- Calling 'get_greeting' Resource ---
Generated Greeting:
Hello, Colab User! This greeting resource works in Colab using MistralAI as a potential LLM source.

--- Calling 'write_a_poem_with_api' Tool (Simulating API) ---
  [SIMULATION] Calling MistralAI API to generate a poem with <= 15 words.
  [SIMULATION] Received response from API.
Generated Poem (<= 15 words, simulated API call):
This is a simulated poem generated via MistralAI API, constrained to 15 words or fewer. Technology flows like streams of light.

--- MCP Server Definition Complete ---
This script defines an MCP server capable of using an external LLM API like MistralAI.
The actual API call within 'write_a_poem_with_api' needs a valid API key and proper LangChain setup.
Remember to add your MistralAI API key to Colab secrets as 

### Demonstration: Simple LLM Call vs. MCP Approach

Let's compare how you would directly interact with an LLM API versus using the MCP framework to call a tool that wraps an LLM API. This highlights the abstraction and standardization provided by MCP.

In [8]:
print("\n--- Demonstrating Simple LLM Call ---")

if LANGCHAIN_AVAILABLE and MISTRAL_API_KEY:
    try:
        # Directly initialize and call the LLM
        llm = ChatMistralAI(model='mistral-small-latest', api_key=MISTRAL_API_KEY, temperature=0.7)
        prompt = "Write a very short, cheerful poem about a sunny day."
        print(f"  [DIRECT LLM] Sending prompt: '{prompt}'")
        direct_llm_response = llm.invoke(prompt)
        print("  [DIRECT LLM] Received response from LLM.")
        print("Direct LLM Poem:")
        print(direct_llm_response.content)
    except Exception as e:
        print(f"An error occurred during direct LLM call: {e}. Please ensure your Mistral_API key is set correctly in Colab secrets.")
else:
    print("Skipping direct LLM call demonstration: LangChain MistralAI not available or API key not found.")
    print("To run this, ensure `Mistral_API` secret is set and LangChain is properly installed and imported.")


--- Demonstrating Simple LLM Call ---
  [DIRECT LLM] Sending prompt: 'Write a very short, cheerful poem about a sunny day.'
  [DIRECT LLM] Received response from LLM.
Direct LLM Poem:
Sunny day,
Sky so bright,
Golden rays,
Pure delight.


In [10]:
print("\n--- Demonstrating MCP Tool Call ---")

# Call the MCP tool defined earlier
try:
    # The MCP tool abstracts away the LLM initialization and API call logic
    mcp_poem = write_a_poem_with_api(word_limit=30)
    print("MCP Tool Poem:")
    print(mcp_poem)
except Exception as e:
    print(f"An error occurred during MCP tool call: {e}")


--- Demonstrating MCP Tool Call ---
  [SIMULATION] Calling MistralAI API to generate a poem with <= 30 words.
  [SIMULATION] Received response from API.
MCP Tool Poem:
This is a simulated poem generated via MistralAI API, constrained to 30 words or fewer. Technology flows like streams of light.
